# 2-4. 모델 평가 기초 실습

이 notebook은 scikit-learn의 실제 분류 모델로 평가 흐름을 확인합니다.

1. train 데이터로 `DecisionTreeClassifier`를 학습합니다.
2. test 데이터의 feature만 넣어 `predict_proba` score를 만듭니다.
3. AUROC/PR-AUC로 score 자체의 품질을 확인합니다.
4. 운영 목적에 맞춰 threshold를 고릅니다.
5. 선택한 threshold로 prediction을 만들고, 마지막에만 test label과 비교합니다.

핵심은 **test label은 채점과 threshold 검토 단계에서만 사용한다**는 점입니다. score나 prediction을 만들 때 label을 쓰면 평가가 아니라 정답을 훔쳐보는 코드가 됩니다.


## 1. 준비

실습용 데이터 로딩은 course helper를 쓰고, 모델 학습과 평가는 scikit-learn API를 직접 사용합니다.


In [ ]:
from __future__ import annotations

import utils as ch02

prepared = await ch02.prepare_notebook()
await ch02.ensure_sklearn()

pd = prepared.pandas
aiq_lite = prepared.aiq_lite

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text

feature_columns = aiq_lite.FEATURE_COLUMNS
target_column = "label"
positive_label = aiq_lite.POSITIVE_LABEL
negative_label = aiq_lite.NEGATIVE_LABEL
configured_threshold = aiq_lite.THRESHOLD
threshold_grid = [round(value / 100, 2) for value in range(5, 100, 5)]

print(f"features={len(feature_columns)}, positive_label={positive_label}")
print(f"configured_threshold={configured_threshold}")


## 2. train/test 데이터 읽기

train 데이터는 모델이 규칙을 배우는 데 사용하고, test 데이터는 학습 이후 품질을 확인하는 데 사용합니다.


In [ ]:
train_raw, train_source = aiq_lite.load_csv_or_sample(
    "data/vital_signs_train.csv",
    aiq_lite.sample_vital_signs(),
    nrows=None,
)
test_raw, test_source = aiq_lite.load_csv_or_sample(
    "data/vital_signs_test.csv",
    aiq_lite.sample_vital_signs(),
    nrows=None,
)

print(f"train: {len(train_raw)} rows from {train_source}")
print(f"test: {len(test_raw)} rows from {test_source}")


### 출력 확인: train 데이터 미리보기


In [ ]:
train_raw[["patient_id", *feature_columns, target_column]].head(3)


### 출력 확인: test 데이터 미리보기


In [ ]:
test_raw[["patient_id", *feature_columns, target_column]].head(3)


## 3. label 정리

원본 label 표기가 섞여 있을 수 있으므로 `high_risk`, `low_risk`로 표준화합니다.


In [ ]:
def labelled_frame(raw_dataframe):
    frame = raw_dataframe[["patient_id", *feature_columns, target_column]].copy()
    frame[target_column] = frame[target_column].map(aiq_lite.normalize_label)
    frame = frame[frame[target_column].isin([positive_label, negative_label])]
    return frame.reset_index(drop=True)

train_dataframe = labelled_frame(train_raw)
test_dataframe = labelled_frame(test_raw)

print(f"train usable rows: {len(train_dataframe)}")
print(f"test usable rows: {len(test_dataframe)}")


### 출력 확인: train label 분포


In [ ]:
train_dataframe[target_column].value_counts().rename_axis("label").to_frame("train_rows")


### 출력 확인: test label 분포


In [ ]:
test_dataframe[target_column].value_counts().rename_axis("label").to_frame("test_rows")


## 4. feature와 target 분리

`X_train`, `X_test`에는 생체 신호 feature만 넣습니다. `y_train`은 지도학습 모델을 학습하기 위한 정답이고, `y_test`는 score 생성에는 사용하지 않습니다.


In [ ]:
X_train = train_dataframe[feature_columns]
y_train = (train_dataframe[target_column] == positive_label).astype(int)

X_test = test_dataframe[feature_columns]
y_test = (test_dataframe[target_column] == positive_label).astype(int)

print(f"X_train={X_train.shape}, y_train={y_train.shape}")
print(f"X_test={X_test.shape}, y_test={y_test.shape}")


### 출력 확인: train feature 결측치


In [ ]:
X_train.isna().sum().rename_axis("feature").to_frame("missing_count")


## 5. Decision Tree 학습

결측치는 `SimpleImputer`가 중앙값으로 채우고, 그 다음 `DecisionTreeClassifier`가 feature와 label의 관계를 학습합니다.


In [ ]:
min_samples_leaf = 20 if len(train_dataframe) >= 200 else 2

model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "tree",
            DecisionTreeClassifier(
                max_depth=4,
                min_samples_leaf=min_samples_leaf,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

# supervised learning에서는 train label을 fit 단계에서만 사용합니다.
model.fit(X_train, y_train)

tree = model.named_steps["tree"]
print(f"trained DecisionTreeClassifier depth={tree.get_depth()}, leaves={tree.get_n_leaves()}")


## 6. test score 만들기

여기서는 `X_test`만 모델에 넣습니다. 아직 threshold도 prediction도 만들지 않습니다.


In [ ]:
probabilities = model.predict_proba(X_test)
positive_class_index = list(model.classes_).index(1)
test_scores = probabilities[:, positive_class_index]

score_preview = pd.DataFrame(
    {
        "patient_id": test_dataframe["patient_id"],
        "score": test_scores,
    }
)

score_preview.head(10)


## 7. AUROC와 PR-AUC로 score 품질 확인

AUROC와 PR-AUC는 특정 threshold 하나에 묶이지 않고, score가 positive row를 negative row보다 얼마나 잘 위에 놓는지 봅니다. PR-AUC는 positive 비율이 낮거나 놓치면 안 되는 케이스를 볼 때 특히 중요합니다.


In [ ]:
positive_rate = y_test.mean()
score_quality = pd.DataFrame(
    [
        {
            "positive_rate_baseline": positive_rate,
            "auroc": roc_auc_score(y_test, test_scores),
            "pr_auc": average_precision_score(y_test, test_scores),
        }
    ]
).round(4)

score_quality


### 출력 확인: precision-recall curve 원자료

`precision_recall_curve`는 threshold를 움직일 때 precision과 recall이 어떻게 바뀌는지 계산해 줍니다.


In [ ]:
curve_precision, curve_recall, curve_thresholds = precision_recall_curve(y_test, test_scores)

pr_curve_frame = pd.DataFrame(
    {
        "threshold": curve_thresholds,
        "precision": curve_precision[:-1],
        "recall": curve_recall[:-1],
    }
).sort_values("threshold")

pr_curve_frame.head(10).round(4)


## 8. 목적에 맞게 threshold 후보 비교

이 실습에서는 `high_risk` 선별을 QA 목적이라고 둡니다. 위험 환자를 놓치는 false negative가 더 비싸므로, 먼저 recall 하한을 정하고 그 안에서 precision이 나은 threshold를 고릅니다.


In [ ]:
def evaluate_threshold(candidate_threshold):
    candidate_flag = (test_scores >= candidate_threshold).astype(int)
    matrix = confusion_matrix(y_test, candidate_flag, labels=[1, 0])
    candidate_tp, candidate_fn = matrix[0]
    candidate_fp, candidate_tn = matrix[1]
    return {
        "threshold": candidate_threshold,
        "precision": precision_score(y_test, candidate_flag, zero_division=0),
        "recall": recall_score(y_test, candidate_flag, zero_division=0),
        "f1": f1_score(y_test, candidate_flag, zero_division=0),
        "false_positive": int(candidate_fp),
        "false_negative": int(candidate_fn),
    }

threshold_frame = pd.DataFrame(
    [evaluate_threshold(candidate) for candidate in threshold_grid]
)

threshold_frame.round(4)


### 출력 확인: 목적별 threshold 선택

한 모델에도 목적이 다르면 threshold가 달라집니다. 아래에서는 screening 목적을 기본 선택으로 사용합니다.


In [ ]:
minimum_recall = 0.90
manual_review_fp_limit = 100

screening_pool = threshold_frame[threshold_frame["recall"] >= minimum_recall]
if screening_pool.empty:
    screening_choice = threshold_frame.sort_values(
        ["recall", "precision", "threshold"], ascending=[False, False, False]
    ).iloc[0]
else:
    screening_choice = screening_pool.sort_values(
        ["precision", "threshold"], ascending=[False, False]
    ).iloc[0]

balanced_choice = threshold_frame.sort_values(
    ["f1", "recall", "precision"], ascending=[False, False, False]
).iloc[0]

capacity_pool = threshold_frame[threshold_frame["false_positive"] <= manual_review_fp_limit]
if capacity_pool.empty:
    capacity_choice = threshold_frame.sort_values(
        ["false_positive", "recall"], ascending=[True, False]
    ).iloc[0]
else:
    capacity_choice = capacity_pool.sort_values(
        ["recall", "precision"], ascending=[False, False]
    ).iloc[0]

threshold_decisions = pd.DataFrame(
    [
        {
            "purpose": "high_risk_screening",
            "rule": f"recall >= {minimum_recall:.2f}, then maximize precision",
            **screening_choice.to_dict(),
        },
        {
            "purpose": "balanced_metric",
            "rule": "maximize f1",
            **balanced_choice.to_dict(),
        },
        {
            "purpose": "manual_review_capacity",
            "rule": f"false_positive <= {manual_review_fp_limit}, then maximize recall",
            **capacity_choice.to_dict(),
        },
    ]
)

threshold_decisions.round(4)


### 선택한 운영 threshold


In [ ]:
selected_threshold = float(screening_choice["threshold"])

print(f"selected_threshold={selected_threshold:.2f}")
print(f"selection_purpose=high_risk_screening")
print(f"configured_threshold_reference={configured_threshold:.2f}")


## 9. 선택한 threshold로 prediction 만들기

운영 기준 threshold 이상이면 `high_risk`, 아니면 `low_risk`로 판단합니다. 이 단계도 label 없이 할 수 있어야 합니다.


In [ ]:
predicted_flag = (test_scores >= selected_threshold).astype(int)
predicted_label = pd.Series(predicted_flag).map({1: positive_label, 0: negative_label})

prediction_preview = pd.DataFrame(
    {
        "patient_id": test_dataframe["patient_id"],
        "score": test_scores,
        "prediction": predicted_label,
    }
)

prediction_preview.head(10)


## 10. label과 비교해서 채점

이제 test label을 꺼내서 prediction과 비교합니다. 이 셀부터가 선택한 운영 threshold의 평가 단계입니다.


In [ ]:
sklearn_matrix = confusion_matrix(y_test, predicted_flag, labels=[1, 0])
tp, fn = sklearn_matrix[0]
fp, tn = sklearn_matrix[1]

confusion_summary = pd.DataFrame(
    [
        {"case": "true_positive", "count": int(tp)},
        {"case": "false_negative", "count": int(fn)},
        {"case": "false_positive", "count": int(fp)},
        {"case": "true_negative", "count": int(tn)},
    ]
)

confusion_summary


### 출력 확인: 선택한 threshold의 sklearn metric


In [ ]:
metrics_summary = pd.DataFrame(
    [
        {
            "threshold": selected_threshold,
            "accuracy": accuracy_score(y_test, predicted_flag),
            "precision": precision_score(y_test, predicted_flag, zero_division=0),
            "recall": recall_score(y_test, predicted_flag, zero_division=0),
            "f1": f1_score(y_test, predicted_flag, zero_division=0),
            "auroc": roc_auc_score(y_test, test_scores),
            "pr_auc": average_precision_score(y_test, test_scores),
        }
    ]
).round(4)

metrics_summary


## 11. 틀린 케이스 확인

오분류만 따로 보면 어떤 환자군에서 모델이 약한지 QA 질문을 만들 수 있습니다.


In [ ]:
review_frame = pd.DataFrame(
    {
        "patient_id": test_dataframe["patient_id"],
        "score": test_scores,
        "prediction": predicted_label,
        "label": test_dataframe[target_column],
    }
)
review_frame["error_type"] = "correct"
review_frame.loc[
    (review_frame["prediction"] == positive_label) & (review_frame["label"] == negative_label),
    "error_type",
] = "false_positive"
review_frame.loc[
    (review_frame["prediction"] == negative_label) & (review_frame["label"] == positive_label),
    "error_type",
] = "false_negative"

mistake_frame = review_frame[review_frame["error_type"] != "correct"]
if mistake_frame.empty:
    mistake_frame = pd.DataFrame([{"message": "no misclassified rows at this threshold"}])

mistake_frame.head(10)


## 12. tree 모델 해석

Decision Tree는 feature 중요도와 상위 규칙을 바로 확인할 수 있습니다. 이 정보는 metric 숫자만 볼 때보다 모델 품질을 설명하기 쉽습니다.


In [ ]:
importance_frame = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": tree.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_frame


### 출력 확인: 상위 tree 규칙


In [ ]:
rule_text = export_text(tree, feature_names=feature_columns, max_depth=3)
print("\n".join(rule_text.splitlines()[:24]))


### 출력 확인: 샘플 1건의 decision path

`decision_path()`는 한 row가 tree 안에서 어떤 node를 지나 leaf까지 갔는지 보여줍니다.


In [ ]:
example_position = int(pd.Series(test_scores).idxmax())
example_patient_id = test_dataframe.iloc[example_position]["patient_id"]
example_features = model.named_steps["imputer"].transform(X_test.iloc[[example_position]])

node_indicator = tree.decision_path(example_features)
leaf_id = int(tree.apply(example_features)[0])
node_ids = node_indicator.indices[node_indicator.indptr[0] : node_indicator.indptr[1]]
leaf_values = tree.tree_.value[leaf_id][0]
leaf_probability = leaf_values[list(tree.classes_).index(1)] / leaf_values.sum()

path_rows = []
for step, node_id in enumerate(node_ids):
    if int(node_id) == leaf_id:
        path_rows.append(
            {
                "patient_id": example_patient_id,
                "step": step,
                "node": int(node_id),
                "feature": "leaf",
                "value": None,
                "threshold": None,
                "decision": f"predict {positive_label} probability={leaf_probability:.3f}",
            }
        )
        continue

    feature_index = int(tree.tree_.feature[node_id])
    threshold_value = float(tree.tree_.threshold[node_id])
    feature_name = feature_columns[feature_index]
    observed_value = float(example_features[0, feature_index])
    direction = "<=" if observed_value <= threshold_value else ">"

    path_rows.append(
        {
            "patient_id": example_patient_id,
            "step": step,
            "node": int(node_id),
            "feature": feature_name,
            "value": round(observed_value, 3),
            "threshold": round(threshold_value, 3),
            "decision": f"{feature_name} {direction} {threshold_value:.3f}",
        }
    )

decision_path_frame = pd.DataFrame(path_rows)
decision_path_frame


## 13. QA 판단 정리

마지막에는 score 품질, threshold 선택 이유, 운영 threshold의 오류 유형을 짧게 요약합니다.


In [ ]:
print(f"auroc={metrics_summary.loc[0, 'auroc']:.4f}, pr_auc={metrics_summary.loc[0, 'pr_auc']:.4f}")
print(f"selected_threshold={selected_threshold:.2f} for high_risk_screening")
print(f"precision={metrics_summary.loc[0, 'precision']:.4f}, recall={metrics_summary.loc[0, 'recall']:.4f}")
print("다음 단계에서는 이 평가 결과를 실험 기록과 데이터 품질 증거에 연결합니다.")
